# Pupil Design for Computational Wavefront Estimation — Demo

Recovering a wavefront from a **single** intensity image is impossible for a symmetric
pupil: the image cannot tell a wavefront from its mirror image. Breaking that symmetry
makes the problem solvable, and this notebook shows how much symmetry-breaking helps.

We will

1. measure the asymmetry of some pupil shapes,
2. see why a symmetric pupil is ambiguous,
3. run a trained network on simulated measurements,
4. reproduce the paper's central result: error falls as asymmetry rises.

Runs on CPU. A GPU makes step 4 faster.

## Setup

The cell below lets the notebook run straight from a fresh clone, with or without
`pip install -e .`.


In [ ]:
# Make the package importable whether or not it has been pip-installed.
# If you ran `pip install -e .` this is a no-op; otherwise it adds ../src.
import sys, pathlib
try:
    import pupil_design
except ModuleNotFoundError:
    root = pathlib.Path.cwd()
    for candidate in [root, *root.parents]:
        src = candidate / "src"
        if (src / "pupil_design").is_dir():
            sys.path.insert(0, str(src))
            break
    else:
        raise ModuleNotFoundError(
            "Could not find pupil_design. Run `pip install -e .` from the repository root."
        )
    import pupil_design
print("pupil_design", pupil_design.__version__, "from", pathlib.Path(pupil_design.__file__).parent)


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from pupil_design import (
    asymmetry, reference_circle, psf_from_pupil, random_zernike_phase,
    generate_polygon_pupils, build_model, simulate_measurement,
    masked_piston_corrected_mse_per_sample, ExperimentConfig,
)

torch.manual_seed(0)
np.random.seed(0)
def pick_device(min_free_gb=1.5):
    """Use the GPU only when it actually has room. Shared machines often do not."""
    import os
    if os.environ.get("DEMO_FORCE_CPU"):
        return torch.device("cpu")
    if not torch.cuda.is_available():
        return torch.device("cpu")
    free_bytes, _ = torch.cuda.mem_get_info()
    if free_bytes / 1e9 < min_free_gb:
        print(f"GPU has only {free_bytes / 1e9:.2f} GB free; falling back to CPU.")
        return torch.device("cpu")
    return torch.device("cuda")

device = pick_device()
print("device:", device)

## 1. Measuring asymmetry

The metric compares a pupil with its own 180-degree rotation and reports how little
they overlap. 0 means perfectly symmetric.

> **Note.** The default `align="published"` reproduces the paper exactly, including a
> known one-pixel alignment error that puts a small positive floor on every value.
> `align="corrected"` fixes it. See the README. We use `"corrected"` here so a circle
> reads 0.

In [ ]:
BBOX, RADIUS = 128, 19

def polygon_pupil(n_sides, radius=RADIUS, bbox=BBOX, rotation=0.0):
    """A regular n-sided aperture, for intuition."""
    from pupil_design.pupils import rasterize_polygon
    angles = np.linspace(0, 2 * np.pi, n_sides, endpoint=False) + rotation
    verts = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    return torch.from_numpy(rasterize_polygon(bbox, verts, radius)).float()

shapes = {
    "circle": reference_circle(BBOX, RADIUS),
    "square": polygon_pupil(4),
    "pentagon": polygon_pupil(5),
    "triangle": polygon_pupil(3),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (name, mask) in zip(axes, shapes.items()):
    a = float(asymmetry(mask, align="corrected"))
    ax.imshow(mask, cmap="gray"); ax.set_title(f"{name}\nasymmetry = {a:.3f}"); ax.axis("off")
plt.tight_layout(); plt.show()

Circles and squares are symmetric under a half turn, so they score 0. Triangles and
pentagons, with odd-numbered sides, are not.

## 2. Why symmetry destroys information

Take a wavefront and its negative. For a symmetric pupil the two produce the **same**
intensity image, so no algorithm can tell them apart. Break the symmetry and the images
differ, which is exactly the information a network needs.

In [ ]:
phase_np = random_zernike_phase(RADIUS, BBOX, num_phases=1, coeff_scale=1.5)[0]
phase = torch.from_numpy(phase_np).float()

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for row, (name, mask) in enumerate([("circle (symmetric)", shapes["circle"]),
                                    ("triangle (asymmetric)", shapes["triangle"])]):
    psf_pos = psf_from_pupil(mask, mask * phase, normalize=True)
    psf_neg = psf_from_pupil(mask, mask * (-phase), normalize=True)
    diff = (psf_pos - psf_neg).abs()

    for ax, img, title in zip(
        axes[row],
        [psf_pos, psf_neg, diff],
        [f"{name}\nPSF of +phase", "PSF of -phase", f"difference (max {diff.max():.1e})"],
    ):
        ax.imshow(img[48:80, 48:80], cmap="inferno"); ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

The circle's difference image is numerically zero: the two wavefronts are
indistinguishable. The triangle's is not, so the sign is recoverable.

## 3. Running a trained model

Point `RUN_DIR` at a directory holding `best_model.pth` and `config.json`. Skip this
section if you have not trained or downloaded weights yet.

In [ ]:
from pathlib import Path
from pupil_design.evaluate import load_run

RUN_DIR = Path("../runs/unet_sigma_0")   # <-- edit me

if RUN_DIR.exists():
    cfg, weights = load_run(RUN_DIR)
    model = build_model(cfg.model_type, cfg.bounding_box_size, cfg.with_pupil, cfg.narrow).to(device)
    model.load_state_dict(torch.load(weights, map_location=device, weights_only=True))
    model.eval()
    print(f"Loaded {cfg.model_type}, noise sigma {cfg.noise_sigma}")
else:
    model = None
    print(f"No run at {RUN_DIR}. Train one with:")
    print("  pupil-design-train --demo --model unet --out runs/demo")

In [ ]:
if model is not None:
    circle_psf = psf_from_pupil(
        reference_circle(cfg.bounding_box_size, cfg.radius, device=device),
        torch.zeros(cfg.bounding_box_size, cfg.bounding_box_size, device=device),
    )
    fig, axes = plt.subplots(3, 4, figsize=(11, 8))
    for col, (name, mask) in enumerate(shapes.items()):
        mask_b = mask.to(device).view(1, 1, BBOX, BBOX)
        target = mask_b * phase.to(device).view(1, 1, BBOX, BBOX)
        psf = simulate_measurement(mask_b, target, circle_psf, cfg)
        with torch.no_grad():
            pred = model(torch.cat([psf, mask_b], dim=1) if cfg.with_pupil else psf).float()

        err = float(masked_piston_corrected_mse_per_sample(pred, target, mask_b))

        def centred(x):
            x = (x[0, 0] * mask_b[0, 0]).cpu().numpy()
            m = mask_b[0, 0].cpu().numpy() > 0
            x[m] -= x[m].mean()
            return np.where(m, x, np.nan)

        lim = np.nanmax(np.abs(centred(target)))
        axes[0, col].imshow(psf[0, 0].cpu()[48:80, 48:80], cmap="inferno")
        axes[0, col].set_title(f"{name}\nmeasurement", fontsize=9)
        axes[1, col].imshow(centred(target), cmap="RdBu_r", vmin=-lim, vmax=lim)
        axes[1, col].set_title("true wavefront", fontsize=9)
        axes[2, col].imshow(centred(pred), cmap="RdBu_r", vmin=-lim, vmax=lim)
        axes[2, col].set_title(f"recovered (MSE {err:.3f})", fontsize=9)
        for r in range(3):
            axes[r, col].axis("off")
    plt.tight_layout(); plt.show()

## 4. The headline result

Sample pupils spanning the asymmetry range, then plot recovery error against asymmetry.
Sampling is binned so that asymmetry is covered evenly and the trend cannot be an
artefact of where the samples happen to fall.

In [ ]:
if model is not None:
    # A U-Net forward pass costs about 2 s per image on CPU and milliseconds on
    # GPU, so scale the sample size to whatever this machine is.
    on_gpu = device.type == "cuda"
    n_pupils, n_phases, n_bins = (60, 16, 10) if on_gpu else (15, 3, 5)
    print(f"Sampling {n_pupils} pupils x {n_phases} phases on {device.type} "
          f"(about {'20 s' if on_gpu else '2 min'})")

    pool = generate_polygon_pupils(radius=cfg.radius, bounding_box_size=cfg.bounding_box_size,
                                   count=n_pupils, n_bins=n_bins)
    phases = torch.from_numpy(
        random_zernike_phase(cfg.radius, cfg.bounding_box_size, num_phases=n_phases)
    ).float().to(device)

    masks = pool["masks"].to(device)
    errors = np.zeros(masks.shape[0])
    with torch.no_grad():
        for i in range(masks.shape[0]):
            m = masks[i].expand(phases.shape[0], -1, -1).unsqueeze(1)
            t = (masks[i].unsqueeze(0) * phases).unsqueeze(1)
            psf = simulate_measurement(m, t, circle_psf, cfg)
            pred = model(torch.cat([psf, m], dim=1) if cfg.with_pupil else psf).float()
            errors[i] = float(masked_piston_corrected_mse_per_sample(pred, t, m).mean())

    asym = pool["asym"]
    fit = np.poly1d(np.polyfit(asym, errors, 1))
    xs = np.linspace(asym.min(), asym.max(), 100)

    plt.figure(figsize=(7, 4.5))
    plt.scatter(asym, errors, s=18, alpha=0.6, label="pupils")
    plt.plot(xs, fit(xs), "r-", lw=2, label=f"trend (slope {fit.c[0]:+.2f})")
    plt.xlabel("Pupil asymmetry"); plt.ylabel("Wavefront MSE (lower is better)")
    plt.title("More asymmetric pupils give better wavefront recovery")
    plt.grid(alpha=0.3); plt.legend(frameon=False); plt.tight_layout(); plt.show()

    print(f"correlation: {np.corrcoef(asym, errors)[0, 1]:+.3f}  (negative means more asymmetry helps)")

## Where to go next

Train the full sweep behind the paper:

```bash
bash scripts/reproduce_paper.sh ./data ./runs
```

Or a single configuration:

```bash
pupil-design-train --model unet --noise-sigma 0.01 --epochs 4 --out runs/unet_0.01
pupil-design-eval runs/unet_0.01 --split test --bins 30
```

Published noise levels are 0, 0.001, 0.01 and 0.03.